# 🚀 Macro Ingestion & Multi-Agent Economic Surprise Calculation

This notebook implements the **Macro Ingestion Strategy** (Roadmap Step 2) using a multi-agent delegation workflow:
1. The user requests a macro surprise calculation for an event (e.g., *CPI MoM* or *Unemployment*).
2. The **Chief Macro Economist Agent** delegates calendar details harvesting to the **ForexFactory Scraper Agent**.
3. The Chief Macro Economist Agent delegates historical standard deviation calculations to the **Alpha Vantage Agent**.
4. The Chief Macro Economist Agent aggregates the metrics, performs the mathematical calculation of the surprise index $\mathcal{S}_t$, and outputs a structured JSON report.

### ⚙️ Step 1: Imports & Environment Configuration

We import core Python libraries (`asyncio`, `json`, `os`, `sys`, etc.) and load environment variables from the `.env.local` file.
We also dynamically append the parent `sentiment` directory to the Python path (`sys.path`) so that Python can resolve custom modules like `functions`.

In [ ]:
import sys
import os
import json
import asyncio
import pandas as pd
import numpy as np
import requests
from dotenv import load_dotenv

# Ensure the sentiment folder is in python path for importing modules
notebook_dir = os.getcwd()
sentiment_dir = os.path.dirname(notebook_dir)
if sentiment_dir not in sys.path:
    sys.path.insert(0, sentiment_dir)

# Load environment variables from .env.local
load_dotenv("../.env.local")

True

### 📦 Step 2: Install Dependencies

Ensure the Model Context Protocol (MCP) Python SDK and Server-Sent Events (SSE) client transport package are installed. These are required to handle connections with the local ForexFactory stdio server and remote Alpha Vantage SSE server.

In [ ]:
# Install MCP SDK if missing (uncomment if running for the first time)
# !pip install mcp sseclient-py

### 🛠️ Step 3: MCP Client Helper Functions

We define async helper functions to connect to and interact with our MCP servers:
- **`async_query_alpha_vantage_mcp`**: Establishes an HTTP connection via **Server-Sent Events (SSE)** to the hosted Alpha Vantage MCP server (`mcp.alphavantage.co`) using your API key. It calls specific tools to retrieve historical macroeconomic datasets.
- **`async_query_forexfactory_mcp`**: Connects via standard input/output (**stdio**) to a local ForexFactory MCP server instance spawned using `npx forexfactory-mcp` to parse economic calendars.

In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from mcp.client.sse import sse_client

async def async_query_alpha_vantage_mcp(tool_name: str, arguments: dict, api_key: str) -> dict:
    """Query the remote Alpha Vantage MCP server using SSE transport.
    
    Use this tool when you need to connect to the hosted Alpha Vantage MCP server 
    to invoke tools for stock data, historical financials, or macroeconomic stats.
    
    Args:
        tool_name (str): The name of the tool to execute on the server.
        arguments (dict): The parameters to pass to the tool.
        api_key (str): Your Alpha Vantage API key.
        
    Returns:
        dict: The parsed JSON response content from the tool.
            In case of failure (connection errors, invalid keys), returns an error payload:
            {"status": "error", "error_msg": str}
    """
    server_url = f"https://mcp.alphavantage.co/mcp?apikey={api_key}"
    try:
        async with sse_client(server_url) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                result = await session.call_tool(tool_name, arguments)
                if isinstance(result.content, list) and len(result.content) > 0:
                    return json.loads(result.content[0].text)
                return {"status": "error", "error_msg": "Empty tool response"}
    except Exception as e:
        return {"status": "error", "error_msg": f"MCP Connection failed: {str(e)}"}


async def async_query_forexfactory_mcp(tool_name: str, arguments: dict) -> dict:
    """Query the local ForexFactory MCP server using stdio transport.
    
    Use this tool when you need to spawn the ForexFactory MCP server locally 
    via stdio to fetch economic calendar event details.
    
    Args:
        tool_name (str): The name of the tool to execute (e.g., 'get_calendar').
        arguments (dict): Parameters for the tool (e.g., {'period': 'today'}).
        
    Returns:
        dict: The parsed economic calendar event data list.
            In case of failure (local process or scraper errors), returns an error payload:
            {"status": "error", "error_msg": str}
    """
    server_params = StdioServerParameters(
        command="npx",
        args=["-y", "forexfactory-mcp"],
        env=None
    )
    try:
        async with stdio_client(server_params) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                result = await session.call_tool(tool_name, arguments)
                if isinstance(result.content, list) and len(result.content) > 0:
                    return json.loads(result.content[0].text)
                return {"status": "error", "error_msg": "Empty tool response"}
    except Exception as e:
        return {"status": "error", "error_msg": f"Local stdio server launch failed: {str(e)}"}

### 🔌 Step 4: Data Ingestion & Mathematical Handlers

We implement the core functions called by our agents to fetch real-world data:
- **`get_forexfactory_economic_calendar`**: Queries the ForexFactory MCP server to retrieve scheduled events, consensus forecasts, and actual releases for a specified period (e.g. `today`).
- **`get_alpha_vantage_historical_std`**: Queries the Alpha Vantage MCP server to fetch historical time-series for a macroeconomic indicator, computes the month-over-month differences, and calculates the rolling historical standard deviation (sigma) over a given window.

**Note**: In alignment with using MCP servers as direct data tools, all simulated/mock fallbacks have been removed. If a tool call fails, an exception is raised to propagate the error cleanly.

In [ ]:
def get_forexfactory_economic_calendar(time_period: str = "today") -> list:
    """Retrieves economic calendar events from ForexFactory for a specified period using ForexFactory MCP.
    
    Args:
        time_period (str, optional): The calendar query window (e.g., 'today', 'tomorrow', 'this_week'). Defaults to 'today'.
        
    Returns:
        list: A list of dicts representing economic events, each containing:
            {
                "event": str,
                "actual": float or None,
                "consensus": float or None,
                "impact": str (red/orange/yellow),
                "currency": str,
                "date": str
            }
    """
    print(f"[*] Fetching economic calendar events for time period: {time_period} via ForexFactory MCP...")
    
    try:
        loop = asyncio.get_event_loop()
    except RuntimeError:
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
    
    mcp_result = loop.run_until_complete(
        async_query_forexfactory_mcp("get_calendar", {"period": time_period})
    )
        
    if isinstance(mcp_result, dict) and mcp_result.get("status") == "error":
        raise RuntimeError(f"ForexFactory MCP tool failed: {mcp_result.get('error_msg')}")
        
    return mcp_result


def get_alpha_vantage_historical_std(indicator: str, window: int = 12) -> float:
    """Retrieves the rolling historical standard deviation of a macroeconomic indicator from Alpha Vantage via MCP.
    
    Args:
        indicator (str): The indicator name matching Alpha Vantage keys (e.g., 'CPI', 'UNEMPLOYMENT', 'RETAIL_SALES').
        window (int, optional): The rolling historical window of months/releases to compute standard deviation over. Defaults to 12.
        
    Returns:
        float: The rolling historical standard deviation of the indicator values.
    """
    api_key = os.getenv("ALPHA_VANTAGE_API_KEY", "").strip('"\' ')
    if not api_key:
        raise ValueError("ALPHA_VANTAGE_API_KEY is not configured in environment.")
        
    print(f"[*] Fetching historical baseline for indicator: {indicator} (Rolling window: {window} periods) via Alpha Vantage MCP...")
    
    try:
        loop = asyncio.get_event_loop()
    except RuntimeError:
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        
    mcp_result = loop.run_until_complete(
        async_query_alpha_vantage_mcp(
            tool_name=indicator.upper(),
            arguments={},
            api_key=api_key
        )
    )
    
    if isinstance(mcp_result, dict) and mcp_result.get("status") == "error":
        raise RuntimeError(f"Alpha Vantage MCP tool failed: {mcp_result.get('error_msg')}")
        
    records = mcp_result.get("data", [])
    if not records:
        raise ValueError(f"No historical records returned for indicator: {indicator}")
        
    # Extract values as floats and compute MoM differences
    values = []
    for record in records[:window+1]:
        val_str = record.get("value", "")
        if val_str and val_str != ".":
            values.append(float(val_str))
            
    if len(values) < 2:
        raise ValueError(f"Insufficient historical data points ({len(values)}) to compute standard deviation.")
        
    # Calculate changes/surprises
    changes = np.diff(values)
    std_val = float(np.std(changes))
    
    if std_val == 0.0:
        raise ValueError("Computed standard deviation is zero; cannot divide by zero.")
        
    return std_val

### ⚖️ Step 5: Macro Surprise Calculation logic

The standardized formula for calculating the macroeconomic surprise score for a given release is:
$$\mathcal{S}_t = w_{tier} \times \frac{|Actual_t - Consensus_t|}{\sigma_{historical}}$$

Where the weight $w_{tier}$ scales the importance of the economic release based on its target impact classification:
- **Red** (High Impact): $1.0$
- **Orange** (Medium Impact): $0.5$
- **Yellow** (Low Impact): $0.2$

We import this calculation function from our utilities and run a quick test calculation to check correctness.

In [9]:
from functions.utils.formulas import calculate_macro_surprise

# Test formula logic
surprise, warning = calculate_macro_surprise(actual=0.6, consensus=0.2, historical_std=0.15, tier="red")
print(f"Test Calculation:")
print(f"  Surprise Score: {surprise:.4f}")
print(f"  Warning Triggered: {warning}")

Test Calculation:
  Surprise Score: 2.6667
  Warning Triggered: False


### 🤖 Step 6: Setup AutoGen & FinRobot Agents

We construct the three specialized agents for this workflow:
1. **ForexFactory Scraper Agent**: Crawls real-time economic calendars and parses details using the ForexFactory MCP server.
2. **Alpha Vantage Agent**: Calculates baseline historical metrics using the Alpha Vantage MCP server.
3. **Chief Macro Economist Agent**: The coordinator agent that orchestrates requests, starts nested chats to collect details, calculates the macro surprise score, and compiles the final JSON report.

System prompt templates are loaded dynamically from file templates in the `prompts` folder to maintain modularity.

In [10]:
import autogen
from finrobot.agents.workflow import FinRobot
from autogen import UserProxyAgent
from functions.utils.config import generate_config
from functions.utils.read_and_clean import read_file_content

# Read LLM configs
nvidia_base_model = os.getenv("NVIDIA_BASE_MODEL", "").strip('"\' ')
nvidia_api_endpoint = os.getenv("NVIDIA_API_ENDPOINT", "https://integrate.api.nvidia.com/v1").strip('"\' ')
nvidia_api_key = os.getenv("NVIDIA_API_KEY", "").strip('"\' ')

hf_api_key = os.getenv("HUGGINGFACE_API_KEY", "").strip('"\' ')
hf_model_name = os.getenv("HUGGINGFACE_MODEL_NAME_FEATHERLESS", "curiousily/Llama-3-8B-Instruct-Finance-RAG").strip('"\' ')
hf_base_url = os.getenv("HUGGINGFACE_BASE_URL", "https://router.huggingface.co/v1").strip('"\' ')

config_list = generate_config(hf_model_name, hf_base_url, hf_api_key)
base_config_list = generate_config(nvidia_base_model, nvidia_api_endpoint, nvidia_api_key)

llm_config = {"config_list": config_list, "model": hf_model_name}
base_llm_config = {"config_list": base_config_list, "model": nvidia_base_model}

# Load external system prompts from the prompts folder
forexfactory_prompt = read_file_content("../prompts/forexfactory_scraper_prompt.txt")
alphavantage_prompt = read_file_content("../prompts/alphavantage_agent_prompt.txt")
chief_macro_economist_prompt = read_file_content("../prompts/chief_macro_economist_prompt.txt")

# Create the UserProxy
user_proxy = UserProxyAgent(
    name="User_Proxy",
    human_input_mode="NEVER",
    is_termination_msg=lambda x: x.get("content", "") and "TERMINATE" in x.get("content", ""),
    max_consecutive_auto_reply=1,
    code_execution_config={"use_docker": False}
)

# Create the sub-agents and orchestrator using FinRobot class
# 1. ForexFactory Scraper Agent
forexfactory_agent = FinRobot(
    agent_config={
        "name": "ForexFactory_Scraper_Agent",
        "description": "Specialist scraper that retrieves real-time calendar values for specific macroeconomic events.",
        "profile": forexfactory_prompt,
        "toolkits": []
    },
    llm_config=llm_config
)

# 2. Alpha Vantage Agent
alphavantage_agent = FinRobot(
    agent_config={
        "name": "AlphaVantage_Agent",
        "description": "Specialist in historical baselines and rolling standard deviation calculations.",
        "profile": alphavantage_prompt,
        "toolkits": []
    },
    llm_config=llm_config
)

# 3. Chief Macro Economist Agent
macro_cio_agent = FinRobot(
    agent_config={
        "name": "Chief_Macro_Economist",
        "description": "Executive macro analyst that calculates final macro surprise scores and compiles JSON reports.",
        "profile": chief_macro_economist_prompt,
        "toolkits": []
    },
    llm_config=base_llm_config
)

d:\PartnaStudio\sentinel\stack\FinRobot-IntentChain\sentiment\venv\Lib\site-packages\flaml\__init__.py:20: UserWarning: flaml.automl is not available. Please install flaml[automl] to enable AutoML functionalities.
  warnings.warn("flaml.automl is not available. Please install flaml[automl] to enable AutoML functionalities.")


### 🔌 Step 7: Tool Registration

We register the python data ingestion helper functions as tools using AutoGen's `register_function` utility. This binds `get_forexfactory_economic_calendar` to the Scraper Agent and `get_alpha_vantage_historical_std` to the Alpha Vantage Agent so that the LLMs can call them as native tools.

In [11]:
# Register tools to the respective agents for AutoGen tool-calling
from autogen import register_function

register_function(
    get_forexfactory_economic_calendar,
    caller=forexfactory_agent.assistant,
    executor=user_proxy,
    name="get_forexfactory_economic_calendar",
    description="Retrieves economic calendar events from ForexFactory for a specified period."
)

register_function(
    get_alpha_vantage_historical_std,
    caller=alphavantage_agent.assistant,
    executor=user_proxy,
    name="get_alpha_vantage_historical_std",
    description="Retrieves rolling historical standard deviation of a macroeconomic indicator from Alpha Vantage."
)

AttributeError: 'FinRobot' object has no attribute 'assistant'

### 🏁 Step 8: Trigger Nested Delegation Chat

We configure the final agent workflow:
1. The `Chief_Macro_Economist` has **nested chats** registered on it. When triggered by the user proxy, it automatically initiates sub-dialogues with the Scraper and Alpha Vantage Agents.
2. The sub-agents fetch the details and return their JSON results.
3. The Chief Macro Economist aggregates the results, executes the calculation, and outputs the structured report.

In [ ]:
from functions.utils.read_and_clean import extract_and_clean_response

# Set up the economic calendar event we want to analyze
target_event = "CPI MoM"
target_indicator = "CPI"

# Configure nested chats on the Macro CIO agent
nested_chats = [
    {
        "recipient": forexfactory_agent,
        "message": lambda recipient, messages, sender, config: (
            f"Please retrieve the latest economic calendar events for today to find the '{target_event}' details."
        ),
        "summary_method": "last_msg",
        "max_turns": 2,
    },
    {
        "recipient": alphavantage_agent,
        "message": lambda recipient, messages, sender, config: (
            f"Please compute and return the rolling historical standard deviation for macro indicator '{target_indicator}' (window 12)."
        ),
        "summary_method": "last_msg",
        "max_turns": 2,
    }
]

macro_cio_agent.register_nested_chats(
    nested_chats,
    trigger=user_proxy
)

print(f"\n[+] Initiating Macro Ingestion delegation workflow for event: '{target_event}'...")
user_proxy.initiate_chat(
    macro_cio_agent,
    message=(
        f"Retrieve economic details for the '{target_event}' event and the historical standard deviation for '{target_indicator}'. "
        "Aggregate them to calculate the macro surprise index S_t and output the final JSON report."
    )
)

final_macro_report = extract_and_clean_response(user_proxy, macro_cio_agent, is_json=True)
print("\n================ FINAL MACRO SURPRISE REPORT ================")
print(final_macro_report)